# Curriculum-to-Skill Matching with Pretrained Embeddings

This notebook reads the curriculum dataset and the industry skill list, embeds both with a sentence-transformers model, and writes matching results to CSV.

Run the cells in order. You can switch the model to `all-MiniLM-L6-v2`, `BAAI/bge-small-en-v1.5`, or `intfloat/e5-base-v2`.

In [ ]:
from pathlib import Path
import os
import re
import csv
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().resolve()
WORKSPACE = ROOT.parent
COURSES_CSV = WORKSPACE / 'curriculum-generator-kb' / 'data' / 'curriculum_dataset_with_ids.csv'
SKILLS_MD = WORKSPACE / 'curriculum-generator-kb' / '03_industry_skills_data.md'
OUTPUT_DIR = ROOT
HF_HOME = ROOT / 'hf_cache'
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

print(f'Notebook folder: {ROOT}')
print(f'Courses: {COURSES_CSV.exists()}')
print(f'Skills: {SKILLS_MD.exists()}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
def build_course_rows(csv_path: Path):
    frame = pd.read_csv(csv_path)
    needed = ['id', 'university', 'program', 'curriculum_year', 'course', 'classification']
    missing = [col for col in needed if col not in frame.columns]
    if missing:
        raise ValueError(f'Course CSV missing columns: {missing}')
    rows = []
    for _, row in frame.iterrows():
        course_id = str(row['id']).strip()
        title = str(row['course']).strip()
        if not course_id or not title:
            continue
        metadata_parts = [
            row.get('university'),
            row.get('program'),
            row.get('curriculum_year'),
            row.get('classification'),
        ]
        metadata = ' | '.join(part for part in (str(p).strip() for p in metadata_parts) if part)
        description = f'{title}. {metadata}' if metadata else title
        rows.append({'id': course_id, 'title': title, 'description': metadata, 'text': description})
    return rows

def parse_skill_rows(md_path: Path):
    if not md_path.exists():
        raise FileNotFoundError(f'Skill markdown not found: {md_path}')
    rows = []
    for line in md_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        l = line.strip()
        if not l.startswith('|') or 'IS-' not in l:
            continue
        cells = [cell.strip() for cell in l.strip('|').split('|')]
        if len(cells) < 11:
            continue
        first = cells[0]
        if not re.match(r'^IS-\d+$', first):
            continue
        skill_id = first
        skill_name = cells[1]
        skill_type = cells[2]
        notes = ' '.join(cells[10:]).strip()
        rows.append({
            'id': skill_id,
            'name': skill_name,
            'type': skill_type,
            'notes': notes,
            'text': f'{skill_name}. {skill_type}. {notes}' if notes else skill_name,
        })
    dedup = {}
    for row in rows:
        dedup.setdefault(row['id'], row)
    return list(dedup.values())

course_rows = build_course_rows(COURSES_CSV)
skill_rows = parse_skill_rows(SKILLS_MD)

print(f'Loaded {len(course_rows)} courses and {len(skill_rows)} skills.')
print(course_rows[0]['text'])
print(skill_rows[0]['text'])

In [ ]:
def resolve_model_name(raw: str) -> str:
    value = raw.strip()
    lower = value.lower()
    if 'bge-small' in lower:
        return 'BAAI/bge-small-en-v1.5'
    if 'bge-base' in lower:
        return 'BAAI/bge-base-en-v1.5'
    if 'minilm' in lower:
        return 'sentence-transformers/all-MiniLM-L6-v2'
    if 'e5' in lower:
        return 'intfloat/e5-base-v2'
    return value

def prefix_text(model_name: str, text: str) -> str:
    if 'e5' in model_name.lower():
        return f'passage: {text}'
    return text

def embed_texts(model: SentenceTransformer, model_name: str, texts):
    prepared = [prefix_text(model_name, text) for text in texts]
    embeddings = model.encode(
        prepared,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    if not isinstance(embeddings, torch.Tensor):
        embeddings = torch.as_tensor(embeddings)
    return embeddings.cpu()

def run_matching(model_name: str, top_k: int = 5, limit: int | None = None):
    model_name = resolve_model_name(model_name)
    course_rows_subset = course_rows[:limit] if limit is not None else course_rows
    skill_rows_subset = skill_rows

    course_texts = [row['text'] for row in course_rows_subset]
    skill_texts = [row['text'] for row in skill_rows_subset]

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(model_name, device=device)
    course_embeddings = embed_texts(model, model_name, course_texts)
    skill_embeddings = embed_texts(model, model_name, skill_texts)
    similarities = torch.matmul(course_embeddings, skill_embeddings.T).numpy()

    course_matches = []
    for i, course in enumerate(course_rows_subset):
        order = sorted(range(len(skill_rows_subset)), key=lambda j: similarities[i, j], reverse=True)[:top_k]
        for rank, j in enumerate(order, start=1):
            course_matches.append({
                'course_id': course['id'],
                'course_title': course['title'],
                'skill_id': skill_rows_subset[j]['id'],
                'skill_name': skill_rows_subset[j]['name'],
                'skill_type': skill_rows_subset[j]['type'],
                'score': float(similarities[i, j]),
                'rank': rank,
            })

    coverage_rows = []
    for j, skill in enumerate(skill_rows_subset):
        best_course_index = int(similarities[:, j].argmax())
        coverage_rows.append({
            'skill_id': skill['id'],
            'skill_name': skill['name'],
            'skill_type': skill['type'],
            'best_course_id': course_rows_subset[best_course_index]['id'],
            'best_course_title': course_rows_subset[best_course_index]['title'],
            'score': float(similarities[best_course_index, j]),
        })

    coverage_rows.sort(key=lambda row: row['score'])
    return course_matches, coverage_rows

In [ ]:
def write_csv(path: Path, fieldnames, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, '') for key in fieldnames})

model_name = 'BAAI/bge-small-en-v1.5'
course_matches, coverage_rows = run_matching(model_name=model_name, top_k=5)

write_csv(
    OUTPUT_DIR / 'course_to_skill_matches.csv',
    ['course_id', 'course_title', 'skill_id', 'skill_name', 'skill_type', 'score', 'rank'],
    course_matches,
)
write_csv(
    OUTPUT_DIR / 'skill_coverage.csv',
    ['skill_id', 'skill_name', 'skill_type', 'best_course_id', 'best_course_title', 'score'],
    coverage_rows,
)

print(f'Model: {model_name}')
print(f'Course matches written: {OUTPUT_DIR / "course_to_skill_matches.csv"}')
print(f'Skill coverage written: {OUTPUT_DIR / "skill_coverage.csv"}')
print('First 5 skill coverage rows:')
for row in coverage_rows[:5]:
    print(row)